# SafeRoads Pothole Detection Model Training (YOLOv8s)
This notebook fine-tunes a **YOLOv8s** (Small) model on the **IVCNZ Real Pothole Dataset** (1,243 annotated road images).

### Instructions:
1. Open this notebook in **Google Colab** with a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).
2. Run **Section 1** to confirm GPU availability (`nvidia-smi`).
3. Run **Section 2** to install dependencies (`ultralytics`).
4. Run **Section 3** to download and prepare the dataset automatically.
5. Run **Section 4** to start 50-epoch training.
6. Run **Section 5** to validate the model and view Precision / Recall / mAP50 metrics.
7. Run **Section 6** to download `best.pt` directly to your computer.
8. Place `best.pt` in `ai-service/models/best.pt` on your local environment.

In [ ]:
# Section 1: Check GPU availability
!nvidia-smi

In [ ]:
# Section 2: Install dependencies
!pip install -q ultralytics pyyaml matplotlib torch torchvision

In [ ]:
# Section 3: Download & Prepare IVCNZ Dataset
import os
import urllib.request
import zipfile
import random
import shutil
from pathlib import Path

DATASET_URL = "https://github.com/jaygala24/pothole-detection/releases/download/v1.0.0/Pothole.Dataset.IVCNZ.zip"
dataset_dir = Path("./dataset")
train_img = dataset_dir / "images" / "train"
val_img = dataset_dir / "images" / "val"
train_lbl = dataset_dir / "labels" / "train"
val_lbl = dataset_dir / "labels" / "val"

for d in [train_img, val_img, train_lbl, val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

yaml_content = f"""path: {dataset_dir.resolve().as_posix()}
train: images/train
val: images/val

names:
  0: pothole
"""
with open(dataset_dir / "data.yaml", "w") as f:
    f.write(yaml_content)

zip_path = Path("./pothole_dataset.zip")
extract_dir = Path("./temp_extract")

print("Downloading IVCNZ dataset...")
urllib.request.urlretrieve(DATASET_URL, zip_path)

print("Extracting archive...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

raw_images = list(extract_dir.rglob("*.jpg")) + list(extract_dir.rglob("*.png"))
pairs = [(img, img.with_suffix(".txt")) for img in raw_images if img.with_suffix(".txt").exists()]
print(f"Found {len(pairs)} image-label pairs.")

random.seed(42)
random.shuffle(pairs)
num_val = int(len(pairs) * 0.2)
val_pairs, train_pairs = pairs[:num_val], pairs[num_val:]

for img, txt in train_pairs:
    shutil.copy(img, train_img / img.name)
    shutil.copy(txt, train_lbl / txt.name)

for img, txt in val_pairs:
    shutil.copy(img, val_img / img.name)
    shutil.copy(txt, val_lbl / txt.name)

shutil.rmtree(extract_dir, ignore_errors=True)
zip_path.unlink(missing_ok=True)
print(f"Dataset prepared: {len(train_pairs)} train images, {len(val_pairs)} val images.")

In [ ]:
# Section 4: Train YOLOv8s Model (50 Epochs)
from ultralytics import YOLO
from pathlib import Path

dataset_dir = Path("./dataset")
model = YOLO('yolov8s.pt')

results = model.train(
    data=str(dataset_dir / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    name='pothole_yolov8s',
    project='runs',
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    save=True,
    plots=True
)

In [ ]:
# Section 5: Validate Model & Display Metrics
metrics = model.val()
print("--------------------------------------------------")
print("          YOLOv8s Pothole Validation Metrics      ")
print("--------------------------------------------------")
try:
    print(f"Precision (mP): {metrics.box.mp * 100:.2f}%")
    print(f"Recall (mR):    {metrics.box.mr * 100:.2f}%")
    print(f"mAP50:          {metrics.box.map50 * 100:.2f}%")
    print(f"mAP50-95:       {metrics.box.map * 100:.2f}%")
except Exception as e:
    print("Results dict:", getattr(metrics, 'results_dict', {}))
print("--------------------------------------------------")

In [ ]:
# Section 6: Download best.pt weights
from google.colab import files
best_weights = Path("runs/pothole_yolov8s/weights/best.pt")
if best_weights.exists():
    print(f"[+] Found best weights at: {best_weights}")
    print("[+] Triggering download to your computer...")
    files.download(str(best_weights))
else:
    print("[!] Weights file not found at:", best_weights)